# CS224N L01 — Co-occurrence Matrix & SVD (mini)

**Code Capsule: cooccurrence-svd-mini (WP02)**

This notebook demonstrates:
1. Building a co-occurrence matrix from a tiny corpus
2. Observing sparsity patterns
3. Applying truncated SVD for dimensionality reduction
4. Visualizing 2D word embeddings

**Official anchor:** notes §3.1; A1 Part 1 cells on co-occurrence matrix and SVD

In [ ]:
import json
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [ ]:
corpus = [
    "word carries meaning",
    "sentence has grammar",
    "language uses grammar",
    "word forms sentence",
    "meaning in language",
    "grammar in sentence",
    "model needs data",
    "train the model",
    "loss drives gradient",
    "gradient updates model",
    "data feeds train",
    "model minimizes loss",
    "learn from data",
    "learn vector meaning",
    "vector from model",
]

for i, sent in enumerate(corpus):
    print(f"  [{i:2d}] {sent}")

In [ ]:
tokenized = [sent.lower().split() for sent in corpus]
vocab_set = set()
for tokens in tokenized:
    vocab_set.update(tokens)
vocab = sorted(vocab_set)
word_to_idx = {w: i for i, w in enumerate(vocab)}
V = len(vocab)
print(f"Vocabulary (size = {V}): {vocab}")

In [ ]:
WINDOW_SIZE = 1
cooccurrence_matrix = [[0] * V for _ in range(V)]
for tokens in tokenized:
    for i, center_word in enumerate(tokens):
        center_idx = word_to_idx[center_word]
        start = max(0, i - WINDOW_SIZE)
        end = min(len(tokens), i + WINDOW_SIZE + 1)
        for j in range(start, end):
            if i == j: continue
            cooccurrence_matrix[center_idx][word_to_idx[tokens[j]]] += 1

total_entries = V * V
nonzero = sum(1 for i in range(V) for j in range(V) if cooccurrence_matrix[i][j] > 0)
print(f"Matrix: {V}x{V} = {total_entries} entries, {nonzero} non-zero, sparsity={100*(total_entries-nonzero)/total_entries:.1f}%")

In [ ]:
M = np.array(cooccurrence_matrix, dtype=np.float64)
K = 2
U, S, Vt = np.linalg.svd(M, full_matrices=False)
U_k, S_k = U[:, :K], S[:K]
word_embeddings = U_k * S_k[np.newaxis, :]

print(f"Singular values: [{S_k[0]:.4f}, {S_k[1]:.4f}]")
print(f"Energy retained: {sum(S_k**2)/sum(S**2)*100:.1f}%")
print()
for i, w in enumerate(vocab):
    print(f"  {w:>14s}  {word_embeddings[i,0]:10.4f}  {word_embeddings[i,1]:10.4f}")

In [ ]:
def cosine_sim(a, b):
    d = np.dot(a, b)
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return float(d/(na*nb)) if na>1e-10 and nb>1e-10 else 0.0

pairs = [("word","meaning","same cluster"), ("model","train","same cluster"),
         ("loss","gradient","same cluster"), ("word","model","cross cluster"),
         ("grammar","loss","cross cluster")]
for w1, w2, note in pairs:
    sim = cosine_sim(word_embeddings[word_to_idx[w1]], word_embeddings[word_to_idx[w2]])
    print(f"  ({w1}, {w2}): {sim:.4f}  ({note})")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 9))
lang_w = {"word","meaning","language","sentence","grammar"}
comp_w = {"model","train","data","loss","gradient"}
for i, w in enumerate(vocab):
    x, y = word_embeddings[i,0], word_embeddings[i,1]
    c, m = ("#2196F3","o") if w in lang_w else ("#FF5722","s") if w in comp_w else ("#4CAF50","^")
    ax.scatter(x, y, s=120, c=c, marker=m, zorder=5, edgecolors="black", linewidths=0.5)
    ax.annotate(w, (x,y), textcoords="offset points", xytext=(8,5), fontsize=10, fontweight="bold")
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([0],[0],marker="o",color="w",markerfacecolor="#2196F3",markersize=10,label="Language"),
    Line2D([0],[0],marker="s",color="w",markerfacecolor="#FF5722",markersize=10,label="Computation"),
    Line2D([0],[0],marker="^",color="w",markerfacecolor="#4CAF50",markersize=10,label="Bridge")], loc="best")
ax.axhline(y=0, color="gray", linewidth=0.5, linestyle="--")
ax.axvline(x=0, color="gray", linewidth=0.5, linestyle="--")
ax.set_xlabel("SVD Dimension 0"); ax.set_ylabel("SVD Dimension 1")
ax.set_title("Co-occurrence SVD: 2D Word Embeddings (window=1, k=2)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()